# [검증] 집계구 유동인구-기온 계절성 — 2026-10-04

- 작성자: 이정연 (`ljy`)
- 목적: 12월 SK 유동인구 감소가 기온(한파) 때문인지, 기온과 무관하게 지속되는 악화인지 1차 진단
- 입력: `data/raw/sk_통신인구_202507~202512_통합.csv`(6개월), `data/raw/reference_data/기상/`의 Open-Meteo ERA5 일별 기온
- 대상 시군구: 강남구(11230, 집계구 기준 "서울")·춘천시(32010, 집계구 기준 "춘천")
- **선행 분석**: `notebooks/260923_EDA_external_factors.ipynb`가 카드·기상·전철을 지역 단위로 결합했다. 이 노트북은 그 다음 단계를 SK 유동인구 + 집계구 단위로 수행한다.
- **한계**: 상관계수 하나로 "순수 계절효과다/아니다"를 확정하지 않는다. 기온도 7~12월 사이 단조 감소하므로, 유동인구가 기온과 무관한 이유로 단조 감소해도 비슷한 상관이 나올 수 있다. 같은 집계구의 6개 관측치는 독립이 아니라서 p-value는 참고용이다. ERA5 서울·춘천 지점은 구·시 전체의 대리변수이지 집계구별 관측이 아니다.

In [1]:
from pathlib import Path

import pandas as pd

from src.data.weather import load_monthly_temperature
from src.features.block_panel import aggregate_to_block, load_oa_boundaries
from src.features.flow_temperature import (
    correlate_flow_with_temperature,
    flow_deviation,
    monthly_log_flow,
    temperature_deviation,
)

MONTHS = ["202507", "202508", "202509", "202510", "202511", "202512"]

repo_root = Path.cwd()
if not (repo_root / "data" / "raw").exists():
    repo_root = repo_root.parent
raw_dir = repo_root / "data" / "raw"

boundaries = load_oa_boundaries(raw_dir / "reference_data")
frames = [
    pd.read_csv(
        raw_dir / f"sk_통신인구_{month}_통합.csv",
        dtype={"STD_YM": "string", "BLOCK_CD": "string"},
    )
    for month in MONTHS
]
combined = pd.concat(frames, ignore_index=True)
panel = aggregate_to_block(combined, boundaries)
print(f"집계구×월 패널 행: {len(panel):,}, 고유 집계구 수: {panel['집계구'].nunique():,}")

집계구×월 패널 행: 10,170, 고유 집계구 수: 1,695


In [2]:
flow = monthly_log_flow(panel)
flow_dev = flow_deviation(flow)

temp = load_monthly_temperature(raw_dir / "reference_data" / "기상")
temp_dev = temperature_deviation(temp)
print("지역별 월평균기온(편차):")
print(temp_dev.round(2).to_string(index=False))

result = correlate_flow_with_temperature(flow_dev, temp_dev)
print("\n지역별 유동인구-기온 편차 Spearman 상관:")
print(result.round(4).to_string())

지역별 월평균기온(편차):
지역 STD_YM  평균기온  평균기온_편차
서울 202507 27.74    10.70
서울 202508 27.33    10.29
서울 202509 22.82     5.78
서울 202510 16.18    -0.86
서울 202511  7.86    -9.18
서울 202512  0.31   -16.73
춘천 202507 27.85    10.55
춘천 202508 27.74    10.43
춘천 202509 22.84     5.54
춘천 202510 16.20    -1.10
춘천 202511  8.14    -9.17
춘천 202512  1.04   -16.26

지역별 유동인구-기온 편차 Spearman 상관:
    correlation  p_value     n
지역                            
서울       0.2719      0.0  6576
춘천       0.1756      0.0  3594


## 결론

- **서울(강남) r=0.272, 춘천 r=0.176 — 둘 다 양(+)의 상관이고 통계적으로 유의하지만, 크기는 약~중간 수준이다.** 기온이 유동인구와 같은 방향으로 움직이는 건 맞다(따뜻할수록 유동인구도 높은 경향) — 하지만 상관이 1에 가깝지 않다는 건, 12월 감소를 기온만으로 설명할 수 없는 부분이 상당히 남아있다는 뜻이다.
- **n이 커서(6,576 / 3,594) p-value가 사실상 0으로 나오지만, 이건 통계적 증거의 강도를 과장해서 보여준다.** 같은 집계구의 6개 관측치는 서로 독립이 아니라서(반복측정), 이 p-value를 "매우 강한 증거"로 읽으면 안 된다. 상관계수(0.17~0.27) 자체가 더 중요한 숫자다.
- **기온-월 공선성 한계는 여전히 남아있다**: 기온도 7월→12월로 단조 감소하고, 유동인구도 전반적으로 감소 경향이 있어서(이전 α 검증 작업에서 추세_위험 평균이 약한 음수였음), 이 상관이 "기온의 직접 효과"인지 "둘 다 그냥 연말로 갈수록 떨어지는 두 시계열"인지 이 분석만으로는 못 가른다.
- **잠정 결론**: 12월 감소를 전부 "한파 때문"으로 돌릴 근거는 약하다(상관이 완전하지 않음). 그렇다고 전부 "진짜 악화"라고 할 근거도 아니다(양의 상관 자체는 유의하게 있음). **"12월 스냅샷이냐 6개월 평균이냐"라는 질문에는, 이 결과가 평균 쪽에 아주 약간 더 힘을 실어준다** — 기온 영향이 일부 섞여 있다는 걸 확인했으니, 12월 단독 스냅샷보다는 여러 달을 보는 쪽이 이 교란을 희석시킨다.
- **다음 확인 필요**: 기온 대신 "월 순서(1~6)"를 바로 상관시켜서, 기온 상관(0.27/0.18)과 거의 같은 크기가 나오는지 비교하면 공선성 문제를 더 구체적으로 짚을 수 있다. 강수·적설도 아직 안 봤다.